# Module 1: Build the Graph

**Overview**

This module adds five hotel documents to the preloaded graph. The build takes about four minutes.

- **Text layer:** `Document` and `Chunk` nodes store the source text. Each `Chunk` also stores a 1024-dimension embedding for search.
- **Fact layer:** `Hotel`, `Room`, `Amenity`, `Policy`, and `Service` nodes store facts for Cypher queries.
- **Connections:** `FROM_DOCUMENT` and `FROM_CHUNK` connect the source text to its facts.

```text
hotel-tokyo-002.txt
    |  one source file, about 7 KB of text
    v
(:Document {source_filename: "hotel-tokyo-002.txt"})
    ^
    |  FROM_DOCUMENT                      text layer
(:Chunk {text, embedding: 1024 floats})
    ^
    |  FROM_CHUNK                         fact layer
(:Hotel {name, address, guest_rating, total_rooms, email, phone})
    |
    +-[:HAS_ROOM]---------> (:Room {type, bed_configuration, max_occupancy, min_rate})
    +-[:OFFERS_AMENITY]---> (:Amenity {name})
    +-[:HAS_POLICY]-------> (:Policy {name, description})
    +-[:PROVIDES_SERVICE]-> (:Service {name, description, cost, hours})
```

Vector search finds text with a similar meaning. Cypher queries match exact facts in the connected nodes. Later modules use both methods.

## Load the shared workshop code

Run the next two cells to load the workshop helpers, read the configuration, and set the AWS Region. Module 1 and Module 2 use the same graph-building code.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module(
    "01-build-graph", extra_import_dirs=("02-connected-context",)
)
print(f"Workshop root: {REPO_ROOT}")

In [ ]:
# Bedrock needs a region, and botocore reads only AWS_DEFAULT_REGION, never
# AWS_REGION. This sets both from one resolved value so that clients built
# without an explicit region_name land in the workshop's region instead of
# whatever the active AWS profile happens to configure.
from workshop.aws_region import configure_aws_region

print(f"Region: {configure_aws_region()}")

## Check the graph before the build

The restored graph contains the main hotel corpus. It does not contain the five documents for this build or the indexes needed for retrieval.

The next cell records the current document and hotel counts. You will compare them with the counts after the build. Each source must connect to one `Hotel`. The build stops when a source has a missing or duplicate `Hotel`, or when two sources connect to the same `Hotel`.

In [ ]:
from graph_builder import connect, count_documents
from workshop.graph_connection import graph_database, require_neo4j_env

require_neo4j_env()

with connect() as driver:
    with driver.session(database=graph_database()) as session:
        hotels_before = session.run("MATCH (h:Hotel) RETURN count(h) AS n").single()["n"]
    documents_before = count_documents(driver)

print(f"Database: {graph_database()}")
print(f"Documents already loaded: {documents_before}")
print(f"Hotels already loaded:    {hotels_before}")

### Inspect a preloaded hotel's text and facts

The next cell shows the text and facts for one preloaded hotel. It reports the chunks, embedding width, rooms, amenities, policies, and services. Use this result as a baseline. Near the end of the notebook, you will run the same check for a hotel you added. This cell only reads data.

In [ ]:
# Read-only. Reports the text and fact layers for one document.
from graph_layers import show_both_layers

show_both_layers("hotel-paris-001.txt")

## Load the five documents for this build

This build uses five reserved documents. They cover Tokyo, Sydney, Rio de Janeiro, Cape Town, and Prague.

- **Reserved files:** Later modules do not depend on these five hotels.
- **Preloaded cities:** The restored graph already contains another hotel for each selected city.
- **Cairo test:** Cairo stays unchanged because Module 2 uses its preloaded data for a retrieval comparison.

The next cell unpacks the files and previews the first document. The preview shows hotel details written as prose. Later sections of the file list rooms, amenities, policies, and services. The build turns this content into graph nodes.

In [ ]:
from held_out_documents import HELD_OUT_DOCUMENTS, extract_held_out

paths = extract_held_out()
for path in paths:
    print(f"  {path.name}  ({path.stat().st_size:,} bytes)")

print(f"\n--- {paths[0].name}, first 400 characters ---")
print(paths[0].read_text(encoding="utf-8")[:400])

## Extract facts and write graph nodes

**Brief overview**

- **Optional comparison:** Run one extraction without a schema and inspect the labels that the model creates.
- **Fixed schema:** Define the node types, relationships, and properties that the model may create.
- **`SimpleKGPipeline`:** Split, embed, extract, and write the prose facts.
- **Amenity parser:** Read exact amenity names from the source list and add them after prose extraction.
- **Build command:** Run both code paths in the required order and check the result.

### Optional: compare labels without the schema

First, look at what happens without a schema. This example is optional.

Set `RUN_UNPINNED_DEMO` to `True` to extract one document without a schema. The cell prints the labels created by the model. It uses model tokens. Leave the value as `False` to skip it. The main build works either way.

The demo uses a temporary source filename. It deletes its temporary graph data when it finishes. It leaves the five build documents and the preloaded graph unchanged.

In [ ]:
# Optional. Extract one document without a schema and report its labels.
RUN_UNPINNED_DEMO = False
UNPINNED_DEMO_SOURCE_FILENAME = "demo-unpinned-schema-comparison.txt"

if RUN_UNPINNED_DEMO:
    from graph_builder import clear_document, session as build_session, snapshot_chunk_ids
    from neo4j_graphrag.components.text_splitters.fixed_size_splitter import (
        FixedSizeSplitter,
    )
    from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

    from graph_config import CHUNK_OVERLAP, CHUNK_SIZE
    from workshop.bedrock_providers import BedrockEmbeddings, BedrockLLM

    sample = paths[0]
    driver = connect()
    try:
        baseline = snapshot_chunk_ids(driver)
        unpinned = SimpleKGPipeline(
            llm=BedrockLLM(),
            driver=driver,
            embedder=BedrockEmbeddings(),
            schema=None,
            text_splitter=FixedSizeSplitter(
                chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
            ),
            from_pdf=False,
            perform_entity_resolution=False,
        )
        await unpinned.run_async(
            file_path=UNPINNED_DEMO_SOURCE_FILENAME,
            text=sample.read_text(encoding="utf-8"),
            document_metadata={
                "source_filename": UNPINNED_DEMO_SOURCE_FILENAME,
            },
        )
        new_chunks = list(snapshot_chunk_ids(driver) - baseline)
        with build_session(driver) as neo4j_session:
            invented = neo4j_session.run(
                """
                MATCH (c:Chunk)<-[:FROM_CHUNK]-(n)
                WHERE elementId(c) IN $ids
                UNWIND [l IN labels(n) WHERE NOT l STARTS WITH '__'] AS label
                RETURN label, count(*) AS count ORDER BY count DESC
                """,
                ids=new_chunks,
            ).values()
        print(f"Temporary labels created without a schema: {invented}")
    finally:
        clear_document(driver, UNPINNED_DEMO_SOURCE_FILENAME)
        driver.close()
else:
    print("Skipped. Set RUN_UNPINNED_DEMO = True to run this optional example.")

## Use a fixed graph schema

A graph schema defines the shape of the graph. It gives the model one vocabulary for every document.

- **Node type:** A label such as `Hotel`, `Room`, `Policy`, or `Service`.
- **Relationship type:** A connection such as `HAS_ROOM`, `HAS_POLICY`, or `PROVIDES_SERVICE`.
- **Pattern:** The allowed start node, relationship, and end node.
- **Property:** A value stored on a node, such as `Hotel.address` or `Hotel.guest_rating`.

Without a schema, the model can create a different graph shape for each document. For example, one document can store the address on `Hotel`. Another can create an `Address` node. A query would need a different pattern for each shape.

The fixed schema prevents this drift. It allows four node types and three relationship types for prose extraction. It rejects extra node types, relationship types, and patterns.

The property descriptions also tell the model how to store values.

- **`address`:** Store the full address on the `Hotel` node. Do not create an `Address` node.
- **`guest_rating`:** Read `4.6` from `4.6/5.0` and store the number as a float.

Module 2 reads `name`, `address`, and `guest_rating` from each `Hotel` node. The fixed schema keeps those fields in the same place.

### Configure `SimpleKGPipeline`

`SimpleKGPipeline` uses the fixed schema to extract prose facts. It runs these steps for each document.

| Step | What it does |
|------|--------------|
| Split | `FixedSizeSplitter` cuts the document into text chunks |
| Embed | Amazon Nova creates a 1024-dimension vector for each `Chunk` |
| Extract | Claude reads the text and returns data that follows the schema |
| Resolve | `perform_entity_resolution=False` keeps hotels with the same name separate |
| Write | The Neo4j writer creates and connects the document, chunk, and fact nodes |

- **Chunk size:** `CHUNK_SIZE` is 12000. Every workshop document fits in one chunk, so the model reads all of its prose in one call.
- **Chunk overlap:** `CHUNK_OVERLAP` is 0 because each document creates one chunk.
- **Response length:** `EXTRACTION_MAX_TOKENS` is 16000 so one hotel's JSON response can finish. A shorter limit can cut off the JSON.

The next cell prints the exact schema passed to `SimpleKGPipeline`.

In [ ]:
from workshop.graph_schema import LLM_EXTRACTION_SCHEMA

print("Node types allowed for prose extraction:")
for node_type in LLM_EXTRACTION_SCHEMA["node_types"]:
    properties = ", ".join(p["name"] for p in node_type.get("properties", []))
    print(f"  :{node_type['label']:<9} {properties}")

print("\nAllowed relationship patterns:")
for start, relationship, end in LLM_EXTRACTION_SCHEMA["patterns"]:
    print(f"  (:{start})-[:{relationship}]->(:{end})")

print("\nExtra graph types allowed:")
for setting in (
    "additional_node_types",
    "additional_relationship_types",
    "additional_patterns",
):
    print(f"  {setting}: {LLM_EXTRACTION_SCHEMA[setting]}")

## Read the amenity list with code

Amenities use a separate code path. `SimpleKGPipeline` does not create `Amenity` nodes. The fixed LLM schema excludes `Amenity` and `OFFERS_AMENITY`.

- **Source:** The bullets directly under `## Hotel Amenities`.
- **Parser:** Reads each bullet and stops at the next Markdown heading.
- **Name:** Stores the exact trimmed bullet text as `Amenity.name`.
- **Shared node:** Uses one `Amenity` node when two hotels use the same exact name.
- **Safety check:** Rejects missing sections, empty lists, duplicate names, and prose inside the list.

This rule prevents prose from creating a false amenity. For example, a sentence that says a pool is unavailable sits outside the list and does not create a Pool amenity.

The next cell reads the amenity lists. It only reads the files. It does not write to Neo4j.

In [ ]:
from graph_builder import parse_amenity_lists

parsed_amenities = parse_amenity_lists(paths)
for parsed in parsed_amenities:
    preview = ", ".join(parsed.names[:3])
    print(f"{parsed.source_filename}: {len(parsed.names)} amenities")
    print(f"  First three: {preview}")

## Build the graph from the five documents

The schema code and the amenity code are separate. `run_additive_build` combines them into one build command so they run in the correct order.

1. Read and validate all five amenity lists.
2. Clear earlier copies of these five documents.
3. Use `SimpleKGPipeline` to split, embed, extract, and write the prose facts.
4. Check that each source created one `Document`, one `Chunk`, and one `Hotel`.
5. Add the parsed amenities to each `Hotel`.
6. Check the schema, amenity names, indexes, and data needed by later modules.

**Brief build notes**

- **Safe rerun:** The command clears only these five source filenames before it starts. It leaves the preloaded documents unchanged.
- **Retry:** The command retries each failed document once. Run the cell again if Bedrock still reports throttling.
- **Required result:** All five documents must load. Every amenity must match its source list.
- **Run time:** Expect about four minutes. The cell prints a line after each document finishes.

In [ ]:
from graph_builder import run_additive_build

exit_code = await run_additive_build(paths, "Module 1: building your five hotels")

if exit_code != 0:
    raise RuntimeError(
        "The build did not finish cleanly. Read the output above, then re-run "
        "this cell; it clears only your five documents before retrying."
    )

## Inspect the workshop indexes

The build already created four indexes. Two indexes search chunk text. Two indexes find documents and hotels by stored values.

| Index | Purpose |
|-------|---------|
| `hotel_chunk_embeddings` | Finds chunks with a similar meaning |
| `hotel_chunk_fulltext` | Finds exact words, hotel names, and numbers in chunk text |
| `workshop_document_source_filename` | Finds a document by its source filename |
| `workshop_hotel_name` | Finds hotels by name |

Vector and full-text search solve different problems. Vector search can find relevant text when the question uses different words. Full-text search can match an exact value such as the postal code `60611`. Module 2 compares these search methods.

Document and query embeddings must use the same model and 1024 dimensions. The workshop sets this configuration in code.

The next cell reads the four indexes. Check that each index is online and targets the expected label and property.

In [ ]:
# Read-only. Reports the four indexes the build just created.
from workshop.retrieval_contract import (
    CHUNK_FULLTEXT_INDEX,
    CHUNK_VECTOR_INDEX,
    DOCUMENT_SOURCE_FILENAME_INDEX,
    HOTEL_NAME_INDEX,
)

INDEX_QUERY = """
SHOW INDEXES YIELD name, type, state, labelsOrTypes, properties, options
WHERE name IN $names
RETURN name, type, state, labelsOrTypes, properties, options
ORDER BY name
"""

with connect() as driver:
    with driver.session(database=graph_database()) as session:
        indexes = session.run(
            INDEX_QUERY,
            names=[
                CHUNK_VECTOR_INDEX,
                CHUNK_FULLTEXT_INDEX,
                DOCUMENT_SOURCE_FILENAME_INDEX,
                HOTEL_NAME_INDEX,
            ],
        ).data()

if not indexes:
    print("No workshop index exists yet. Run the build cell above.")

for index in indexes:
    config = (index["options"] or {}).get("indexConfig", {})
    labels = ", ".join(index["labelsOrTypes"])
    properties = ", ".join(index["properties"])
    print(index["name"])
    print(f"  type:       {index['type']}")
    print(f"  state:      {index['state']}")
    print(f"  indexes:    :{labels}({properties})")
    if index["type"] == "VECTOR":
        print(f"  dimensions: {config.get('vector.dimensions')}")
        print(f"  similarity: {config.get('vector.similarity_function')}")

## Check the hotels you added

The next cell lists the five hotels you added. It shows their addresses, ratings, and amenity counts. It also compares document and hotel counts before and after the build.

- **Rating:** Each rating is a number on the `Hotel` node. Later queries can average these values.
- **Amenity count:** Each count comes from the exact bullets under `## Hotel Amenities` in the source file.

The following read-only cell shows the text and fact layers for the first hotel you added. It uses the same check as the preloaded hotel near the start of this notebook.

In [ ]:
with connect() as driver:
    with driver.session(database=graph_database()) as session:
        print("The hotels you just extracted:\n")
        for record in session.run(
            """
            MATCH (d:Document)<-[:FROM_DOCUMENT]-(:Chunk)<-[:FROM_CHUNK]-(h:Hotel)
            WHERE d.source_filename IN $filenames
            OPTIONAL MATCH (h)-[:OFFERS_AMENITY]->(a:Amenity)
            WHERE a.name IS NOT NULL
            WITH h, count(DISTINCT a) AS amenities
            RETURN h.name AS name, h.address AS address,
                   h.guest_rating AS rating, amenities
            ORDER BY name
            """,
            filenames=list(HELD_OUT_DOCUMENTS),
        ):
            print(f"  {record['name']}")
            print(f"    {record['address']}")
            print(f"    rating {record['rating']}, {record['amenities']} amenities\n")

        hotels_after = session.run("MATCH (h:Hotel) RETURN count(h) AS n").single()["n"]

    documents_after = count_documents(driver)

print(f"Documents: {documents_before} -> {documents_after}")
print(f"Hotels:    {hotels_before} -> {hotels_after}")
print(
    "\nEvery aggregation and connected traversal from here on runs across the whole "
    "graph, yours included."
)

In [ ]:
# Read-only. Reports both layers for a hotel you added.
show_both_layers(HELD_OUT_DOCUMENTS[0])

## Check a shared amenity node

The preloaded graph contains two Chicago source files. These files are separate from the five documents you added.

Both Chicago files include the exact bullet `Complimentary High-Speed Wifi`. The parser connects both hotels to one shared `Amenity` node. The next read-only cell confirms that both source filenames reach the same node.

In [ ]:
# Read-only. Proves that two source Hotels traverse to one shared Amenity node.
CHICAGO_SOURCES = ["hotel-chicago-001.txt", "hotel-chicago-002.txt"]
CHICAGO_WIFI_QUERY = """
CYPHER 25
MATCH (document:Document)<-[:FROM_DOCUMENT]-(:Chunk)<-[:FROM_CHUNK]-(hotel:Hotel)
MATCH (hotel)-[:OFFERS_AMENITY]->(amenity:Amenity {name: $amenity_name})
WHERE document.source_filename IN $filenames
RETURN elementId(amenity) AS amenity_node_id,
       collect(DISTINCT hotel.name) AS hotels,
       collect(DISTINCT document.source_filename) AS source_filenames
"""

with connect() as driver:
    with driver.session(database=graph_database()) as session:
        shared_wifi = session.run(
            CHICAGO_WIFI_QUERY,
            filenames=CHICAGO_SOURCES,
            amenity_name="Complimentary High-Speed Wifi",
        ).data()

if len(shared_wifi) != 1 or set(shared_wifi[0]["source_filenames"]) != set(CHICAGO_SOURCES):
    raise RuntimeError("The two Chicago Hotels do not share the authored WiFi node.")

print(shared_wifi[0])

## Continue to Module 2

**Purpose:** Use this graph for GraphRAG retrieval.

You now have searchable source chunks, connected hotel facts, and vector and full-text indexes. Module 2 compares GraphRAG read paths over this graph.

In [ ]:
from workshop.workshop_utils import lego_progress

lego_progress(1)